# 04 Real Data Evaluation

This notebook evaluates motif discovery methods on real biological datasets.

For each dataset, the workflow is:

- load the FASTA file
- inspect sequence content and lengths
- create a MEME-safe FASTA copy with simple headers
- run custom algorithms multiple times
- run MEME once
- extract discovered consensuses
- compare all methods to the JASPAR reference motif using Hamming distance and runtime

In [ ]:
import sys
import os
import time
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

project_root = os.path.abspath("..")
sys.path.append(project_root)

from src.fasta_utils import (
    read_fasta,
    save_fasta_with_headers,
    make_meme_safe_headers
)
from src.data_utils import hamming
from src.random_search import random_search
from src.rmhc import rmhc
from src.ga import genetic_algorithm
from src.aco import aco_motif_search
from src.simulated_annealing import simulated_annealing

## MEME Execution Helper Functions

This cell defines helper functions for running MEME through WSL.

- windows_to_wsl_path() converts a Windows file path into a WSL-compatible path.
- run_meme() runs MEME on a FASTA file, constraining the motif width to the known reference length, and records the runtime.

In [ ]:
def windows_to_wsl_path(path): # Convert a Windows file path into WSL format so MEME can access it
    path = os.path.abspath(path)
    drive = path[0].lower()
    rest = path[2:].replace("\\", "/")
    return f"/mnt/{drive}{rest}"


def run_meme(fasta_file, k, output_dir):
    # Convert input and output paths to WSL-compatible format
    wsl_fasta_file = windows_to_wsl_path(fasta_file)
    wsl_output_dir = windows_to_wsl_path(output_dir)
    # Build MEME command for DNA motif discovery using fixed motif width k
    cmd = [
        "wsl",
        os.environ.get("MEME_BIN", "meme"),
        wsl_fasta_file,
        "-dna",
        "-mod", "oops",
        "-nmotifs", "1",
        "-minw", str(k),
        "-maxw", str(k),
        "-oc", wsl_output_dir
    ]
# Execute MEME and measure total runtime
    start = time.time()
    subprocess.run(cmd, check=True)
    runtime = time.time() - start
    return runtime

## Extracting a Consensus Motif from MEME Output

MEME outputs a position probability matrix rather than a direct consensus string.

This cell defines functions to:

- convert a position probability matrix into a consensus motif
- extract the first discovered motif matrix from the `meme.txt` output file

The consensus is formed by selecting the nucleotide with the highest probability at each motif position.

In [ ]:
def pwm_to_consensus(pwm):
    # Convert a probability matrix into a consensus motif by selecti the highest-probability nucleotide at each position
    letters = ["A", "C", "G", "T"]
    consensus = ""

    for row in pwm:
        max_index = row.index(max(row))
        consensus += letters[max_index]

    return consensus


def extract_meme_consensus(meme_txt_file):
    pwm = []
    reading_matrix = False

    with open(meme_txt_file, "r") as f:
        for line in f:
            line = line.strip()
 # Detect the start of MEME's letter-probability matrix
            if "letter-probability matrix" in line:
                reading_matrix = True
                continue
 # Stop when the matrix section ends
            if reading_matrix:
                if not line or line.startswith("-"):
                    break

                parts = line.split()
# Each matrix row contains probabilities for A, C, G, and T
                if len(parts) >= 4:
                    row = [float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])]
                    pwm.append(row)

    if not pwm:
        raise ValueError(f"Could not extract PWM from {meme_txt_file}")
# Convert the extracted matrix into a consensus motif string
    return pwm_to_consensus(pwm)

## Running the Custom Motif Discovery Algorithms

This cell defines a function that runs all custom motif discovery algorithms on the real dataset.

The algorithms are each executed multiple times because they are stochastic methods.  
For every run, the following are recorded:

- algorithm name
- run number
- discovered consensus motif
- runtime

This allows later comparison of both accuracy and efficiency.

In [ ]:
def run_custom_algorithms_real(sequences, k, runs=5):
    results = []
# Execute each stochastic algorithm multiple times to reduce the effect
 # of random variation on the final comparison
    for run in range(runs):
        start = time.time()
        _, _, consensus = random_search(sequences, k, iterations=5000)
        runtime = time.time() - start
        results.append(("Random Search", run, consensus, runtime))

        start = time.time()
        _, _, consensus = rmhc(sequences, k, iterations=5000)
        runtime = time.time() - start
        results.append(("RMHC", run, consensus, runtime))

        start = time.time()
        _, _, consensus = genetic_algorithm(
            sequences,
            k,
            population_size=30,
            generations=200
        )
        runtime = time.time() - start
        results.append(("GA", run, consensus, runtime))

        start = time.time()
        _, _, consensus = aco_motif_search(
            sequences,
            k,
            num_ants=20,
            iterations=100
        )
        runtime = time.time() - start
        results.append(("ACO", run, consensus, runtime))

        start = time.time()
        _, _, consensus = simulated_annealing(
            sequences,
            k,
            iterations=5000,
            initial_temperature=10.0,
            cooling_rate=0.995
        )
        runtime = time.time() - start
        results.append(("SA", run, consensus, runtime))

    return results

## Dataset 1 — Arnt (MA0004.1)

In [ ]:
# Select the real dataset and define the reference motif used for comparison
dataset_name = "MA0004.1.fasta"
tf_name = "Arnt"
reference_motif = "CACGTG"
k = len(reference_motif)
# Load the real FASTA dataset
real_fasta_file = f"../data/real_fasta/{dataset_name}"

headers, real_sequences = read_fasta(real_fasta_file)

print("Loaded", len(real_sequences), "sequences from:", real_fasta_file)
print("Reference motif:", reference_motif)
print("Motif length k:", k)

## Inspecting Sequence Lengths and Validity

Before running the algorithms, the dataset is checked to ensure that it is suitable for motif discovery.

This includes:

- counting the number of sequences
- examining the sequence lengths
- verifying that all sequences contain only valid DNA nucleotides (A, C, G, and T)

This step helps confirm that the dataset has been loaded correctly and contains no unexpected characters.

In [ ]:
# Inspect sequence lengths and check that all symbols are valid DNA bases
lengths = [len(seq) for seq in real_sequences]

print("Number of sequences:", len(real_sequences))
print("Shortest sequence length:", min(lengths))
print("Longest sequence length:", max(lengths))
print("All sequence lengths:", lengths)
print()

valid_dna = {"A", "C", "G", "T"}

for i, seq in enumerate(real_sequences):
    bad_chars = sorted(set(seq) - valid_dna)
    if bad_chars:
        print(f"Sequence {i} has non-DNA characters:", bad_chars)
    else:
        print(f"Sequence {i} contains only valid DNA characters (A/C/G/T)")

## Preparing MEME Input and Initialising Result Storage

This cell creates the folders needed to save MEME outputs, result tables, and plots.

The original FASTA headers are then converted into MEME-safe headers, and a cleaned FASTA file is saved for use with MEME.

Finally, an empty results list is created to store the motifs and runtimes produced by each algorithm during the experiment.

In [ ]:
# Create folders for MEME outputs, result tables, plots, and processed FASTA files
os.makedirs("../results/real/meme", exist_ok=True)
os.makedirs("../results/real/tables", exist_ok=True)
os.makedirs("../results/real/plots", exist_ok=True)
os.makedirs("../data/real_fasta", exist_ok=True)
# Replace original FASTA headers with MEME-safe headers before running MEME
safe_headers = make_meme_safe_headers(headers, prefix=tf_name)
meme_ready_file = f"../data/real_fasta/{dataset_name.replace('.fasta', '')}_meme_ready.fasta"

save_fasta_with_headers(safe_headers, real_sequences, meme_ready_file)

print("Saved MEME-ready FASTA to:", meme_ready_file)

## Running the Custom Motif Discovery Algorithms

The custom algorithms are now executed on the MA0004.1 dataset.

Each stochastic method is run multiple times in order to capture variation in performance. For every run, the discovered consensus motif, Hamming distance from the reference motif, and runtime are recorded.

This allows both accuracy and computational efficiency to be compared fairly across methods.

In [ ]:
# Run the custom algorithms on the real dataset and compare their discovered
# consensuses with the reference motif
real_results = []

algo_results = run_custom_algorithms_real(real_sequences, k, runs=5)

for method, run, consensus, runtime in algo_results:
    distance = hamming(consensus, reference_motif)

    real_results.append({
        "dataset": dataset_name,
        "method": method,
        "run": run,
        "consensus": consensus,
        "reference_motif": reference_motif,
        "hamming_distance": distance,
        "runtime": runtime
    })

## Running MEME and Extracting Its Consensus Motif

This cell runs MEME on the prepared FASTA file using the known motif width.

After MEME finishes, the motif probability matrix is extracted from the output file and converted into a consensus sequence. The Hamming distance between MEME’s discovered motif and the known reference motif is then calculated and added to the results.

In [ ]:
# Run MEME on the processed FASTA file and evaluate its discovered motif
meme_output_dir = f"../results/real/meme/{dataset_name.replace('.fasta', '')}_output"
meme_runtime = run_meme(meme_ready_file, k, meme_output_dir)

meme_txt_file = os.path.join(meme_output_dir, "meme.txt")
meme_consensus = extract_meme_consensus(meme_txt_file)
meme_distance = hamming(meme_consensus, reference_motif)

real_results.append({
    "dataset": dataset_name,
    "method": "MEME",
    "run": 0,
    "consensus": meme_consensus,
    "reference_motif": reference_motif,
    "hamming_distance": meme_distance,
    "runtime": meme_runtime
})

print("MEME consensus:", meme_consensus)
print("MEME Hamming distance:", meme_distance)

In [ ]:
## Create result tables

In [ ]:
# Store all real dataset results in a DataFrame for inspection
df_real_1 = pd.DataFrame(real_results)
display(df_real_1)

In [ ]:
# Compute average Hamming distance and runtime for each method on this dataset
summary_real_1 = df_real_1.groupby(["dataset", "method"], as_index=False).agg(
    avg_hamming=("hamming_distance", "mean"),
    avg_runtime=("runtime", "mean")
)

summary_real_1

In [ ]:
# Save summary results for the MA0004.1 dataset
summary_real_1.to_csv("../results/real/tables/MA0004.1_summary.csv", index=False)
print("Saved to ../results/real/tables/MA0004.1_summary.csv")

In [ ]:
# Plot average motif recovery accuracy for each method on MA0004.1

plt.figure(figsize=(8, 5))

plt.bar(summary_real_1["method"], summary_real_1["avg_hamming"])

plt.xlabel("Method")
plt.ylabel("Average Hamming distance")
plt.title("Motif recovery accuracy on MA0004.1")
plt.grid(True, axis="y")
plt.savefig("../results/real/plots/MA0004.1_hamming_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot average runtime for each method on MA0004.1

plt.figure(figsize=(8, 5))
plt.bar(summary_real_1["method"], summary_real_1["avg_runtime"])
plt.xlabel("Method")
plt.ylabel("Average runtime (seconds)")
plt.title("Runtime on MA0004.1")
plt.grid(True, axis="y")
plt.savefig("../results/real/plots/MA0004.1_runtime_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## Dataset 2 — Prrx2 (MA0075.1)

In [ ]:
dataset_name = "MA0075.1.fasta"
tf_name = "Prrx2"
reference_motif = "AATTA"
k = len(reference_motif)

# Load the FASTA sequences for this transcription factor dataset

real_fasta_file = f"../data/real_fasta/{dataset_name}"

headers, real_sequences = read_fasta(real_fasta_file)

print("Loaded", len(real_sequences), "sequences from:", real_fasta_file)
print("Reference motif:", reference_motif)
print("Motif length k:", k)

In [ ]:
# Inspect sequence lengths and validate that the first 10 sequences
# contain only standard DNA characters
lengths = [len(seq) for seq in real_sequences]

print("Number of sequences:", len(real_sequences))
print("Shortest sequence length:", min(lengths))
print("Longest sequence length:", max(lengths))
print("First 10 sequence lengths:", lengths[:10])
print()

valid_dna = {"A", "C", "G", "T"}

for i, seq in enumerate(real_sequences[:10]):
    bad_chars = sorted(set(seq) - valid_dna)
    if bad_chars:
        print(f"Sequence {i} has non-DNA characters:", bad_chars)
    else:
        print(f"Sequence {i} contains only valid DNA characters (A/C/G/T)")

In [ ]:
## Create a MEME-safe FASTA copy (change the headers)
safe_headers = make_meme_safe_headers(headers, prefix=tf_name)
meme_ready_file = f"../data/real_fasta/{dataset_name.replace('.fasta', '')}_meme_ready.fasta"

save_fasta_with_headers(safe_headers, real_sequences, meme_ready_file)

print("Saved MEME-ready FASTA to:", meme_ready_file)

In [ ]:
# Store results for all methods on the MA0075.1 dataset
real_results_2 = []

In [ ]:
## Run the custom algorithms

In [ ]:
# Run the custom algorithms multiple times on the real dataset
# and compare each discovered consensus with the reference motif
algo_results = run_custom_algorithms_real(real_sequences, k, runs=5)

for method, run, consensus, runtime in algo_results:
    distance = hamming(consensus, reference_motif)

    real_results_2.append({
        "dataset": dataset_name,
        "method": method,
        "run": run,
        "consensus": consensus,
        "reference_motif": reference_motif,
        "hamming_distance": distance,
        "runtime": runtime
    })


In [ ]:
## Run MEME once

In [ ]:
# Run MEME on the prepared FASTA file and evaluate its discovered motif
meme_output_dir = f"../results/real/meme/{dataset_name.replace('.fasta', '')}_output"
meme_runtime = run_meme(meme_ready_file, k, meme_output_dir)

meme_txt_file = os.path.join(meme_output_dir, "meme.txt")
meme_consensus = extract_meme_consensus(meme_txt_file)
meme_distance = hamming(meme_consensus, reference_motif)

real_results_2.append({
    "dataset": dataset_name,
    "method": "MEME",
    "run": 0,
    "consensus": meme_consensus,
    "reference_motif": reference_motif,
    "hamming_distance": meme_distance,
    "runtime": meme_runtime
})

print("MEME consensus:", meme_consensus)
print("MEME Hamming distance:", meme_distance)

In [ ]:
## store results in Result tables

In [ ]:
df_real_2 = pd.DataFrame(real_results_2)
display(df_real_2)

In [ ]:
# Compute average Hamming distance and runtime for each method on the MA0075.1 dataset
summary_real_2 = df_real_2.groupby(["dataset", "method"], as_index=False).agg(
    avg_hamming=("hamming_distance", "mean"),
    avg_runtime=("runtime", "mean")
)

display(summary_real_2)

In [ ]:
# Save the summary table 
summary_real_2.to_csv("../results/real/tables/MA0075.1_summary.csv", index=False)
print("Saved to ../results/real/tables/MA0075.1_summary.csv")

In [ ]:
# Plot average motif recovery accuracy 
plt.figure(figsize=(10, 5))
plt.bar(summary_real_2["method"], summary_real_2["avg_hamming"])
plt.xlabel("Method")
plt.ylabel("Average Hamming distance")
plt.title("Motif recovery accuracy on MA0075.1")
plt.xticks(rotation=45)
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig("../results/real/plots/MA0075.1_hamming_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot average runtime 
plt.figure(figsize=(10, 5))
plt.bar(summary_real_2["method"], summary_real_2["avg_runtime"])
plt.xlabel("Method")
plt.ylabel("Average runtime (seconds)")
plt.title("Runtime on MA0075.1")
plt.xticks(rotation=45)
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig("../results/real/plots/MA0075.1_runtime_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## Dataset 3 — MNB1A (MA0053.1)

In [ ]:
dataset_name = "MA0053.1.fasta"
tf_name = "MNB1A"
reference_motif = "AAAGC"
k = len(reference_motif)

real_fasta_file = f"../data/real_fasta/{dataset_name}"

headers, real_sequences = read_fasta(real_fasta_file)

print("Loaded", len(real_sequences), "sequences from:", real_fasta_file)
print("Reference motif:", reference_motif)
print("Motif length k:", k)


In [ ]:
# Inspect sequence lengths and validate that the first 10 sequences contain only standard DNA characters
lengths = [len(seq) for seq in real_sequences]

print("Number of sequences:", len(real_sequences))
print("Shortest sequence length:", min(lengths))
print("Longest sequence length:", max(lengths))
print("First 10 sequence lengths:", lengths[:10])
print()

valid_dna = {"A", "C", "G", "T"}

for i, seq in enumerate(real_sequences[:10]):
    bad_chars = sorted(set(seq) - valid_dna)
    if bad_chars:
        print(f"Sequence {i} has non-DNA characters:", bad_chars)
    else:
        print(f"Sequence {i} contains only valid DNA characters (A/C/G/T)")

In [ ]:
# Convert FASTA headers into MEME-safe format and save a cleaned input file
safe_headers = make_meme_safe_headers(headers, prefix=tf_name)
meme_ready_file = f"../data/real_fasta/{dataset_name.replace('.fasta', '')}_meme_ready.fasta"

save_fasta_with_headers(safe_headers, real_sequences, meme_ready_file)

print("Saved MEME-ready FASTA to:", meme_ready_file)

In [ ]:
# Run the custom algorithms and MEME on the dataset
# then compare discovered consensuses with the reference motif
real_results_3 = []

algo_results = run_custom_algorithms_real(real_sequences, k, runs=5)

for method, run, consensus, runtime in algo_results:
    distance = hamming(consensus, reference_motif)

    real_results_3.append({
        "dataset": dataset_name,
        "method": method,
        "run": run,
        "consensus": consensus,
        "reference_motif": reference_motif,
        "hamming_distance": distance,
        "runtime": runtime
    })

meme_output_dir = f"../results/real/meme/{dataset_name.replace('.fasta', '')}_output"
meme_runtime = run_meme(meme_ready_file, k, meme_output_dir)

meme_txt_file = os.path.join(meme_output_dir, "meme.txt")
meme_consensus = extract_meme_consensus(meme_txt_file)
meme_distance = hamming(meme_consensus, reference_motif)

real_results_3.append({
    "dataset": dataset_name,
    "method": "MEME",
    "run": 0,
    "consensus": meme_consensus,
    "reference_motif": reference_motif,
    "hamming_distance": meme_distance,
    "runtime": meme_runtime
})

print("MEME consensus:", meme_consensus)
print("MEME Hamming distance:", meme_distance)

In [ ]:
# store results
df_real_3 = pd.DataFrame(real_results_3)
display(df_real_3)

In [ ]:
# Compute average Hamming distance and runtime for each method

summary_real_3 = df_real_3.groupby(["dataset", "method"], as_index=False).agg(
    avg_hamming=("hamming_distance", "mean"),
    avg_runtime=("runtime", "mean")
)

display(summary_real_3)

In [ ]:
summary_real_3.to_csv("../results/real/tables/MA0053.1_summary.csv", index=False)
print("Saved to ../results/real/tables/MA0053.1_summary.csv")

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(summary_real_3["method"], summary_real_3["avg_hamming"])
plt.xlabel("Method")
plt.ylabel("Average Hamming distance")
plt.title("Motif recovery accuracy on MA0053.1")
plt.xticks(rotation=45)
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig("../results/real/plots/MA0053.1_hamming_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(summary_real_3["method"], summary_real_3["avg_runtime"])
plt.xlabel("Method")
plt.ylabel("Average runtime (seconds)")
plt.title("Runtime on MA0053.1")
plt.xticks(rotation=45)
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig("../results/real/plots/MA0053.1_runtime_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## Dataset 4 — MEF2A (MA0052.1)

In [ ]:
# Select the fourth real dataset and define its reference motif
dataset_name = "MA0052.1.fasta"
tf_name = "MEF2A"
reference_motif = "CTATTTATAG"
k = len(reference_motif)

real_fasta_file = f"../data/real_fasta/{dataset_name}"

headers, real_sequences = read_fasta(real_fasta_file)

print("Loaded", len(real_sequences), "sequences from:", real_fasta_file)
print("Reference motif:", reference_motif)
print("Motif length k:", k)

In [ ]:
# Inspect sequence lengths and validate the first 10 sequences
lengths = [len(seq) for seq in real_sequences]

print("Number of sequences:", len(real_sequences))
print("Shortest sequence length:", min(lengths))
print("Longest sequence length:", max(lengths))
print("First 10 sequence lengths:", lengths[:10])
print()

valid_dna = {"A", "C", "G", "T"}

for i, seq in enumerate(real_sequences[:10]):
    bad_chars = sorted(set(seq) - valid_dna)
    if bad_chars:
        print(f"Sequence {i} has non-DNA characters:", bad_chars)
    else:
        print(f"Sequence {i} contains only valid DNA characters (A/C/G/T)")

In [ ]:
# Prepare MEME-safe FASTA file
safe_headers = make_meme_safe_headers(headers, prefix=tf_name)
meme_ready_file = f"../data/real_fasta/{dataset_name.replace('.fasta', '')}_meme_ready.fasta"

save_fasta_with_headers(safe_headers, real_sequences, meme_ready_file)

print("Saved MEME-ready FASTA to:", meme_ready_file)

In [ ]:
# Run the custom algorithms and MEME on the dataset
real_results_4 = []

algo_results = run_custom_algorithms_real(real_sequences, k, runs=5)

for method, run, consensus, runtime in algo_results:
    distance = hamming(consensus, reference_motif)

    real_results_4.append({
        "dataset": dataset_name,
        "method": method,
        "run": run,
        "consensus": consensus,
        "reference_motif": reference_motif,
        "hamming_distance": distance,
        "runtime": runtime
    })

meme_output_dir = f"../results/real/meme/{dataset_name.replace('.fasta', '')}_output"
meme_runtime = run_meme(meme_ready_file, k, meme_output_dir)

meme_txt_file = os.path.join(meme_output_dir, "meme.txt")
meme_consensus = extract_meme_consensus(meme_txt_file)
meme_distance = hamming(meme_consensus, reference_motif)

real_results_4.append({
    "dataset": dataset_name,
    "method": "MEME",
    "run": 0,
    "consensus": meme_consensus,
    "reference_motif": reference_motif,
    "hamming_distance": meme_distance,
    "runtime": meme_runtime
})

print("MEME consensus:", meme_consensus)
print("MEME Hamming distance:", meme_distance)

In [ ]:
# Store results in a DataFrame
df_real_4 = pd.DataFrame(real_results_4)
display(df_real_4)

In [ ]:
# Compute summary statistics
summary_real_4 = df_real_4.groupby(["dataset", "method"], as_index=False).agg(
    avg_hamming=("hamming_distance", "mean"),
    avg_runtime=("runtime", "mean")
)

display(summary_real_4)

In [ ]:
# Save summary table
summary_real_4.to_csv("../results/real/tables/MA0052.1_summary.csv", index=False)
print("Saved to ../results/real/tables/MA0052.1_summary.csv")

In [ ]:
# Plot average Hamming distance
plt.figure(figsize=(10, 5))
plt.bar(summary_real_4["method"], summary_real_4["avg_hamming"])
plt.xlabel("Method")
plt.ylabel("Average Hamming distance")
plt.title("Motif recovery accuracy on MA0052.1")
plt.xticks(rotation=45)
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig("../results/real/plots/MA0052.1_hamming_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot average runtime
plt.figure(figsize=(10, 5))
plt.bar(summary_real_4["method"], summary_real_4["avg_runtime"])
plt.xlabel("Method")
plt.ylabel("Average runtime (seconds)")
plt.title("Runtime on MA0052.1")
plt.xticks(rotation=45)
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig("../results/real/plots/MA0052.1_runtime_plot.png", dpi=300, bbox_inches="tight")
plt.show()